# SnapUGC-LightKD Final 300-Video Run

This notebook runs the clean v2 thesis pipeline on a bounded Kaggle subset designed to finish before 24h:

- DOVER-Mobile quality scoring
- CLIP/R(2+1)D/YAMNet/Sentence-T5/Qwen2.5-VL feature extraction
- Temporal Teacher, Student baseline, and Student+KD training


In [ ]:
# 0. Clone this repository into Kaggle working directory
GITHUB_REPO = "https://github.com/TranTop2806/SnapUGC-LightKD.git"
REPO_DIR = "/kaggle/working/SnapUGC-LightKD"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
    %cd /kaggle/working

print("Repo ready:", REPO_DIR)


In [ ]:
# 1. Install dependencies
!pip install -q -U "transformers>=4.49.0" accelerate sentence-transformers qwen-vl-utils av
!pip install -q eva-decord tensorflow tensorflow_hub scipy pandas tqdm matplotlib

import os, sys, glob, json, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. Paths and bounded 300-video subset
MAX_VIDEOS = 300          # keep 200-400 to finish safely before Kaggle timeout
RUN_DOVER = True          # uses DOVER-Mobile for speed
RUN_QWEN = True           # Qwen2.5-VL-3B offline caption/rationale
RESET_FEATURES = True     # avoid resuming from a failed/empty previous run
QWEN_DEVICE = 'cuda:1' if torch.cuda.is_available() and torch.cuda.device_count() > 1 else ('cuda:0' if torch.cuda.is_available() else 'cpu')

INPUT_DIR = '/kaggle/input/datasets/nguyntuncng/snapugc-dataset'
TRAIN_CSV_ORIG = os.path.join(INPUT_DIR, 'train_data.csv')
TRAIN_VIDEO_ROOT = os.path.join(INPUT_DIR, 'train_videos', 'train_videos')
if not os.path.exists(TRAIN_CSV_ORIG):
    raise FileNotFoundError(TRAIN_CSV_ORIG)
if not os.path.isdir(TRAIN_VIDEO_ROOT):
    raise FileNotFoundError(TRAIN_VIDEO_ROOT)

REPO_CANDIDATES = [
    globals().get('REPO_DIR', '/kaggle/working/SnapUGC-LightKD'),
    '/kaggle/working/SnapUGC-LightKD',
]
REPO_DIR = next((p for p in REPO_CANDIDATES if p and os.path.exists(os.path.join(p, 'scripts'))), None)
if REPO_DIR is None:
    raise FileNotFoundError('Cannot find SnapUGC-LightKD. Check the clone cell or update REPO_DIR.')

OUTPUT_DIR = f'/kaggle/working/final_{MAX_VIDEOS}_videos'
SUBSET_VIDEO_DIR = os.path.join(OUTPUT_DIR, 'subset_videos')
SUBSET_CSV = os.path.join(OUTPUT_DIR, f'train_subset_{MAX_VIDEOS}.csv')
FEATURES_PATH = os.path.join(OUTPUT_DIR, f'features_final_{MAX_VIDEOS}.json')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
DOVER_DIR = os.path.join(OUTPUT_DIR, 'dover')
DOVER_CSV = os.path.join(DOVER_DIR, 'dover_scores.csv')
QWEN_JSON = os.path.join(OUTPUT_DIR, 'qwen_captions.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOVER_DIR, exist_ok=True)
os.makedirs(SUBSET_VIDEO_DIR, exist_ok=True)

# Build a stable subset of CSV rows whose videos exist. DOVER and extraction use this subset only.
import pandas as pd, shutil
df = pd.read_csv(TRAIN_CSV_ORIG)
selected_rows = []
for _, row in df.iterrows():
    vid = str(row['Id'])
    src_video = os.path.join(TRAIN_VIDEO_ROOT, f'{vid}.mp4')
    if not os.path.exists(src_video):
        continue
    dst_video = os.path.join(SUBSET_VIDEO_DIR, f'{vid}.mp4')
    if not os.path.exists(dst_video):
        try:
            os.symlink(src_video, dst_video)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(src_video, dst_video)
    selected_rows.append(row)
    if len(selected_rows) >= MAX_VIDEOS:
        break
if len(selected_rows) == 0:
    raise RuntimeError('No matching videos found for subset.')
pd.DataFrame(selected_rows).to_csv(SUBSET_CSV, index=False)

TRAIN_CSV = SUBSET_CSV
TRAIN_VIDEO_DIR = SUBSET_VIDEO_DIR
if RESET_FEATURES:
    for stale_path in [FEATURES_PATH, DOVER_CSV, QWEN_JSON]:
        if os.path.exists(stale_path):
            os.remove(stale_path)

print('INPUT_DIR:', INPUT_DIR)
print('TRAIN_CSV:', TRAIN_CSV)
print('TRAIN_VIDEO_DIR:', TRAIN_VIDEO_DIR)
print('Subset videos:', len(glob.glob(os.path.join(TRAIN_VIDEO_DIR, '*.mp4'))))
print('REPO_DIR:', REPO_DIR)
print('FEATURES_PATH:', FEATURES_PATH)
print('QWEN_DEVICE:', QWEN_DEVICE)
print('QWEN_JSON:', QWEN_JSON)


In [ ]:
# 3. DOVER-Mobile quality scores for the bounded subset
# DOVER-Mobile is the official lightweight DOVER variant and is fast enough for 200-400 videos.
if RUN_DOVER:
    !git clone -q https://github.com/VQAssessment/DOVER.git /kaggle/working/DOVER 2>/dev/null || true
    %cd /kaggle/working/DOVER
    !pip install -q scikit-video yacs timm einops opencv-python-headless decord pyyaml pandas scipy tqdm
    !mkdir -p pretrained_weights
    !wget -q -nc https://github.com/QualityAssessment/DOVER/releases/download/v0.5.0/DOVER-Mobile.pth -O pretrained_weights/DOVER-Mobile.pth
    !ls -lh pretrained_weights
    !python evaluate_a_set_of_videos.py -in "$TRAIN_VIDEO_DIR" -out "$DOVER_CSV" -o dover-mobile.yml
    print('DOVER_CSV:', DOVER_CSV)
    if not os.path.exists(DOVER_CSV):
        raise RuntimeError(f'DOVER did not produce expected CSV: {DOVER_CSV}')
    import pandas as pd
    dover_preview = pd.read_csv(DOVER_CSV)
    print('DOVER rows:', len(dover_preview))
    print(dover_preview.head())
    if len(dover_preview) < max(1, int(MAX_VIDEOS * 0.8)):
        raise RuntimeError(f'DOVER produced too few rows: {len(dover_preview)}/{MAX_VIDEOS}')
else:
    raise RuntimeError('RUN_DOVER must be True for this final bounded run.')

%cd /kaggle/working
print('DOVER_CSV:', DOVER_CSV)


In [ ]:
# 4. Qwen captions/rationales for exactly 300 videos
# This step is isolated from feature extraction. If Qwen fails, the run stops instead of skipping samples.
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/extract_qwen.py'),
    '--csv', TRAIN_CSV,
    '--videos', TRAIN_VIDEO_DIR,
    '--out', QWEN_JSON,
    '--max', str(MAX_VIDEOS),
    '--num-frames', '8',
    '--batch-size', '25',
    '--qwen-device', QWEN_DEVICE,
    '--cpu-fallback',
]
if not RUN_QWEN:
    raise RuntimeError('RUN_QWEN must be True for this final bounded run.')
print(' '.join(cmd))
!{" ".join(cmd)}

with open(QWEN_JSON, 'r', encoding='utf-8') as f:
    qwen_rows = json.load(f)
qwen_count = len(qwen_rows) if isinstance(qwen_rows, dict) else len(qwen_rows)
qwen_non_empty = sum(1 for row in (qwen_rows.values() if isinstance(qwen_rows, dict) else qwen_rows) if str(row.get('qwen_caption', '')).strip())
print(f'Qwen captions: {qwen_non_empty}/{qwen_count}')
if qwen_count != MAX_VIDEOS or qwen_non_empty != MAX_VIDEOS:
    raise RuntimeError(f'Qwen must produce exactly {MAX_VIDEOS} non-empty captions, got {qwen_non_empty}/{qwen_count}.')


In [ ]:
# 5. Feature extraction using precomputed Qwen captions
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/extract_features.py'),
    '--csv', TRAIN_CSV,
    '--videos', TRAIN_VIDEO_DIR,
    '--out', FEATURES_PATH,
    '--max', str(MAX_VIDEOS),
    '--save-every', '25',
    '--num-frames', '16',
    '--motion-clips', '4',
    '--motion-frames', '16',
    '--qwen-json', QWEN_JSON,
    '--strict-qwen',
    '--max-errors-before-abort', '1',
]
if DOVER_CSV:
    cmd += ['--dover-csv', DOVER_CSV]
if not RUN_QWEN:
    raise RuntimeError('RUN_QWEN must be True for this final bounded run.')
print(' '.join(cmd))
!{" ".join(cmd)}


In [ ]:
# 6. Validate feature schema
with open(FEATURES_PATH, 'r', encoding='utf-8') as f:
    features = json.load(f)
print('samples:', len(features))
if len(features) != MAX_VIDEOS:
    raise RuntimeError(f'Expected exactly {MAX_VIDEOS} successful samples, got {len(features)}.')

qwen_ok = sum(1 for x in features if str(x.get('qwen_caption', '')).strip())
dover_ok = sum(
    1 for x in features
    if bool((x.get('dover_scores') or {}).get('found', False))
)
print(f'Qwen non-empty captions: {qwen_ok}/{len(features)}')
print(f'DOVER matched scores: {dover_ok}/{len(features)}')
if RUN_QWEN and qwen_ok != len(features):
    raise RuntimeError(f'Every sample must have a Qwen caption, got {qwen_ok}/{len(features)}.')
if RUN_DOVER and dover_ok != len(features):
    raise RuntimeError(f'Every sample must have a matched DOVER score, got {dover_ok}/{len(features)}.')

first = features[0]
for key in ['clip_frame_embeddings', 'motion_clip_embeddings', 'dover_scores', 'yamnet_embedding_mean', 'metadata_text_embedding', 'caption_embedding', 'rationale_embedding', 'qwen_caption']:
    value = first.get(key)
    if isinstance(value, list):
        print(key, 'len:', len(value), 'nested:', len(value[0]) if value and isinstance(value[0], list) else '')
    else:
        print(key, value if key == 'dover_scores' else type(value))
print('First Qwen caption:', first.get('qwen_caption', '')[:300])


In [ ]:
# 7. Train Teacher, Student baseline, and Student+KD
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/train.py'),
    '--data', FEATURES_PATH,
    '--save-dir', RESULTS_DIR,
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--teacher-epochs', '30',
    '--student-epochs', '40',
    '--batch', '16',
    '--teacher-hidden', '512',
    '--student-hidden', '256',
    '--teacher-blocks', '2',
    '--alpha', '0.5',
    '--beta', '0.3',
    '--attn-kd', '0.1',
    '--gamma', '0.2',
    '--delta', '0.2',
]
print(' '.join(cmd))
!{" ".join(cmd)}


In [ ]:
# 8. Show report
report_path = os.path.join(RESULTS_DIR, 'final_experiment_report.json')
with open(report_path, 'r') as f:
    report = json.load(f)
print(json.dumps(report, indent=2)[:5000])

In [ ]:
# 9. Package outputs
zip_path = f'/kaggle/working/snapugc_lightkd_final_{MAX_VIDEOS}_outputs.zip'
!zip -qr "$zip_path" "$OUTPUT_DIR"
print(zip_path)
